<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign4/NLP_assign4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 - Sentiment Analysis
---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import seaborn as sns

In [2]:
# Download and extract IMDb Large Movie Review Dataset

import os
import tarfile
import urllib.request

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
filename = "aclImdb_v1.tar.gz"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)  # ~80MB, may take a minute

if not os.path.exists("aclImdb"):
    with tarfile.open(filename) as tar:
        tar.extractall(filter='data')

In [3]:
# Loading dataset into a Pandas DataFrame
import pandas as pd

def load_reviews(split):
    data = []
    for label in ['pos', 'neg']:
        folder = os.path.join('aclImdb', split, label)
        for fname in os.listdir(folder):
            if fname.endswith('.txt'):
                with open(os.path.join(folder, fname), encoding='utf-8') as f:
                    data.append((f.read(), label))
    return pd.DataFrame(data, columns=['review', 'sentiment'])

train_df = load_reviews('train')
test_df = load_reviews('test')

train_df

,review,sentiment
0,Cannot believe a movie that can be made that g...,pos
1,"OK, I don't really think that Trailer Park Boy...",pos
2,A very well made film set in early '60s commun...,pos
3,I found this to be a profoundly amusing dark c...,pos
4,The three main characters are very well portra...,pos
...,...,...
24995,"Two years before he wrote and directed ""Arthur...",neg
24996,I literally fell asleep 3 times watching this ...,neg
24997,Without effective indulgence of the supernatur...,neg
24998,I will admit that I did not give this movie mu...,neg


In [4]:
print(f"Training shape: {train_df.shape}")
print(f"Testing shape: {train_df.shape}")

Training shape: (25000, 2)
Testing shape: (25000, 2)


In [5]:
train_df['sentiment'].value_counts()

,count
sentiment,
pos,12500
neg,12500


In [6]:
X_train, y_train = train_df['review'], train_df['sentiment']
X_test, y_test = test_df['review'], test_df['sentiment']

Using current versions of scikit-learn, pandas, nltk/spaCy, and matplotlib/seaborn, solve the below questions.

## Q1: Text Vectorization
---


 - Convert text into numerical features using Bag-of-Words (CountVectorizer), TF-IDF (TfidfVectorizer), and modern vectorization tools.
    - Explain, in your own words, the difference between Bag-of-Words and TF-IDF representations. Which one do you expect to perform better for sentiment analysis, and why?

In [7]:
# Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(stop_words='english', max_features=10000)
X_train_cv = count_vec.fit_transform(X_train)
X_test_cv = count_vec.transform(X_test)

train_dtm = pd.DataFrame(X_train_cv.toarray(), columns=count_vec.get_feature_names_out())
train_dtm.index = X_train.index
train_dtm.head()

,00,000,01,10,100,1000,101,11,12,13,...,zizek,zodiac,zombi,zombie,zombies,zone,zoo,zoom,zorro,zu
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

train_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
train_tfidf.index = X_train.index
train_tfidf.head()

,00,000,01,10,100,1000,101,11,12,13,...,zizek,zodiac,zombi,zombie,zombies,zone,zoo,zoom,zorro,zu
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.073887,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.104624,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Q2: Word Cloud
---

 - Generate a word cloud for positive reviews and negative reviews. Comment on the most frequent words observed in each class.

## Q3: Sentiment Classification
---

 - Train and evaluate a Naive Bayes classifier and a Logistic Regression classifier for sentiment classification.

## Q4: Evaluation Metrics
---

 - Compare the two models using standard evaluation metrics. Build a comparison table summarizing Accuracy, Precision, Recall, F1-score, and training time for:
    - Naive Bayes + Bag-of-Words
    - Naive Bayes + TF-IDF
    - Logistic Regression + TF-IDF

## Review
---

Based on your results, answer the following:
<ol type="a">
    <li>Which model performed better overall?</li>
    <li>Which model is more interpretable, and why?</li>
    <li>Which model would you deploy in a production system? Justify your answer.</li>
</ol>